In [2]:
# SENTIMENT CLASSIFICATION USING AUTOENCODER

import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.metrics import accuracy_score, classification_report

# Load IMDB dataset
vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# Make all reviews same length
x_train = pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = pad_sequences(x_test, maxlen=max_len, padding='post')

# Normalize
x_train = x_train.astype('float32') / vocab_size
x_test = x_test.astype('float32') / vocab_size

# -------- Autoencoder --------
input_layer = Input(shape=(max_len,))

# Encoder
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
latent = Dense(32, activation='relu', name='latent_layer')(encoded)

# Decoder
decoded = Dense(64, activation='relu')(latent)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(max_len, activation='sigmoid')(decoded)

autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer='adam', loss='mse')

# Train Autoencoder
autoencoder.fit(
    x_train, x_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# -------- Extract latent features --------
encoder = Model(input_layer, latent)

X_train_features = encoder.predict(x_train, verbose=0)
X_test_features = encoder.predict(x_test, verbose=0)

# -------- Sentiment Classifier --------
classifier_input = Input(shape=(32,))
x = Dense(16, activation='relu')(classifier_input)
output = Dense(1, activation='sigmoid')(x)

classifier = Model(classifier_input, output)

classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

classifier.fit(
    X_train_features, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# -------- Prediction --------
y_prob = classifier.predict(X_test_features, verbose=0)
y_pred = (y_prob > 0.5).astype(int).ravel()

# -------- Results --------
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Negative', 'Positive']
))

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0459 - val_loss: 0.0243
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0242 - val_loss: 0.0243
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0242 - val_loss: 0.0243
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0241 - val_loss: 0.0243
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0241 - val_loss: 0.0242
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5071 - loss: 0.6963 - val_accuracy: 0.5070 - val_loss: 0.6941
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5052 - loss: 0.6944 - val_accuracy: 0.5032 - val_loss: 0.6941
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5076 - loss: 0.6938 - val_accuracy: 0.5186 - val_loss: 0.6926
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5129 - loss: 0.6928 - val_accuracy: 0.4974 - val_loss: 0.6944
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.51